# 16 · Cooling down in time ☕

Pour a fresh, hot coffee and watch it cool — now on the **same 3D mug** as the
stationary problem (notebook 8). The **transient heat equation**
$$ \partial_t T - \nabla\!\cdot(\kappa\nabla T) = 0,\qquad
   -\kappa\,\partial_n T = \alpha\,(T-T_\infty)\ \text{ on }\Gamma $$
is discretised in space by finite elements and in time by **implicit Euler**:
$$ (M + \Delta t\,K)\,T^{n+1} = M\,T^{n} + \Delta t\,b . $$

In [ ]:
# --- Google Colab: install NGSolve on first run (a no-op anywhere else) -------
# NGSolve ships its PyPI wheels as pre-releases, so the `--pre` flag is essential.
import sys
if "google.colab" in sys.modules:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "--pre",
                    "ngsolve", "webgui_jupyter_widgets"], check=True)

In [ ]:
from netgen.occ import *
from ngsolve import *
from ngsolve.webgui import Draw
import sys

def progress(i, n, label="working"):
    """A tiny dependency-free progress bar (survives JupyterLite / Colab / local)."""
    import os
    if os.environ.get("WEBGUI_SCENE_DIR"):             # static-site build: stay silent (no \r spam in the HTML)
        return
    if (i + 1) % max(1, n // 100) == 0 or i + 1 == n:
        f = int(26 * (i + 1) / n)
        sys.stdout.write(f"\r  {label}… [{'█'*f}{'·'*(26-f)}] {100*(i+1)//n:3d}%")
        sys.stdout.flush()
        if i + 1 == n:
            sys.stdout.write("\n")

def coffee_cup_3d():
    """The ceramic mug with a ring handle and a body of coffee (refined inside)."""
    R, ri, Hout, base, fill = 4.0, 3.4, 9.0, 1.0, 6.0
    outer  = Cylinder(Pnt(0, 0, 0), Z, r=R, h=Hout, bottom="bottom", mantle="wall")
    cavity = Cylinder(Pnt(0, 0, base), Z, r=ri, h=Hout)
    ceramic = outer - cavity
    ceramic.faces.Max(Z).name = "rim"
    # A C-shaped handle swept along an arc with `Pipe`. This is a single solid with no
    # closed-revolve seam, which avoids *both* the JupyterLite OpenCASCADE seam-vertex
    # error ("BRep_Tool: TopoDS_Vertex hasn't gp_Pnt") of a full 360° torus *and* the CI
    # mesher crash that a fused two-half torus triggered.
    zmid, bulge, harm, rt = Hout/2, 2.0, 1.8, 0.45      # mid-height, out-reach, half-height, tube radius
    p1, p2, p3 = Pnt(R-0.15, 0, zmid-harm), Pnt(R+bulge, 0, zmid), Pnt(R-0.15, 0, zmid+harm)
    spine = Wire([ArcOfCircle(p1, p2, p3)])             # bottom attach → bulge → top attach
    section = WorkPlane(Axes(p1, n=Dir(bulge+0.15, 0, harm), h=Y)).Circle(0, 0, rt).Face()
    handle = Pipe(spine, section) - cavity              # trim the part poking inside
    handle.faces.name = "handle"
    ceramic = ceramic + handle
    ceramic.solids.name = "ceramic"
    coffee = Cylinder(Pnt(0, 0, base), Z, r=ri, h=fill - base, top="surface")
    coffee.solids.name = "coffee"
    coffee.maxh = 0.8                                  # resolve the stirred hot spot
    return Glue([ceramic, coffee])

mesh = Mesh(OCCGeometry(coffee_cup_3d()).GenerateMesh(maxh=1.5))
mesh.Curve(2)
clip = {"Clipping": {"enable": True, "function": True, "x": 0, "y": 1, "z": 0, "dist": 0}}
print(f"{mesh.ne} elements, {mesh.nv} vertices")

## 1. Matrices

The conductivity $\kappa$ is **piecewise** (ceramic vs coffee) and the cooling
rate $\alpha$ is a **surface** field (strong at the coffee surface, weaker on the
wall). We assemble the **mass** $M$ and the **stiffness + cooling** $K$ once,
build $M^\ast=M+\Delta t\,K$ and factorise it a single time — every step is then
one back-substitution. $M^\ast$ is symmetric, so `sparsecholesky` fits.

In [ ]:
kappa = mesh.MaterialCF({"ceramic": 1.5, "coffee": 0.6})
alpha = mesh.BoundaryCF({"wall": 6.0, "surface": 10.0}, default=1.5)
T_air, dt, tend = 20.0, 0.5, 15.0

fes = H1(mesh, order=1)
u, v = fes.TnT()
M = BilinearForm(u*v*dx).Assemble()
K = BilinearForm(kappa*grad(u)*grad(v)*dx + alpha*u*v*ds).Assemble()
b = LinearForm(alpha*T_air*v*ds).Assemble()           # cooling source

mstar = M.mat.CreateMatrix()
mstar.AsVector().data = M.mat.AsVector() + dt * K.mat.AsVector()
invmstar = mstar.Inverse(fes.FreeDofs(), inverse="sparsecholesky")

## 2. Time stepping

Start from a fresh, uniformly hot coffee and march in time, storing each frame so
we can **animate** the cool-down (press play after clicking the scene; the view
is clipped so you see *inside* the mug).

In [ ]:
gfu = GridFunction(fes)
gfu.Set(mesh.MaterialCF({"coffee": T_air + 60, "ceramic": T_air + 8}))   # hot coffee

times, temps = [0.0], [gfu(mesh(0, 0, 3))]
gfu.AddMultiDimComponent(gfu.vec)                     # frame 0
t, step, nsteps = 0.0, 0, round(tend / dt)
while t < tend - 1e-9:
    gfu.vec.data = invmstar * (M.mat * gfu.vec + dt * b.vec)
    t += dt
    times.append(t); temps.append(gfu(mesh(0, 0, 3)))
    gfu.AddMultiDimComponent(gfu.vec)
    progress(step, nsteps, "cooling"); step += 1

print(f"coffee centre cooled from {temps[0]:.0f} °C to {temps[-1]:.0f} °C in {tend:.0f} units")
Draw(gfu, mesh, interpolate_multidim=True, animate=True, settings=clip)

## 3. Newton's law of cooling

The centre temperature decays (almost) exponentially towards the air temperature
— Newton's law of cooling, recovered automatically.

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(6, 3))
plt.plot(times, temps, "o-")
plt.axhline(T_air, ls="--", color="grey", label="air")
plt.xlabel("time"); plt.ylabel("centre temperature [°C]"); plt.legend()
plt.tight_layout(); plt.show()

## 4. Teaser — what if we *stir*?

Stirring adds **transport**: a prescribed velocity $\mathbf{w}$ (here a gentle
rotation about the axis, only in the coffee) carries the heat around via the
convection term $\mathbf{w}\!\cdot\!\nabla T$.

A subtle point: putting that convection into the *implicit* matrix would make it
**non-symmetric**, and `sparsecholesky` (which assumes symmetry) would silently
return nonsense. So we treat convection **explicitly** (an IMEX step) — the
implicit operator stays symmetric — and add a touch of **streamline diffusion**
(also symmetric) to keep the sharp hot spot from oscillating:
$$ M^\ast\,T^{n+1} = M\,T^{n} - \Delta t\,C\,T^{n},\qquad
   M^\ast = M + \Delta t\,(K_{\text{diff}} + \text{streamline}). $$

In [ ]:
w = CF((-y, x, 0))                                    # rotational stir, in the coffee
dxc = dx(definedon=mesh.Materials("coffee"))
C = BilinearForm(w*grad(u)*v * dxc).Assemble()        # convection (applied explicitly)

Kstir = BilinearForm(0.15*grad(u)*grad(v)*dx          # mild physical diffusion
                     + 0.25*(w*grad(u))*(w*grad(v))*dxc).Assemble()   # streamline diffusion
ms = M.mat.CreateMatrix()
ms.AsVector().data = M.mat.AsVector() + dt * Kstir.mat.AsVector()
invms = ms.Inverse(fes.FreeDofs(), inverse="sparsecholesky")          # symmetric → OK

gfu.Set(T_air + 60*exp(-((x - 1.6)**2 + y**2 + (z - 3.0)**2) / 1.6))  # off-centre hot spot
gfu.AddMultiDimComponent(gfu.vec)
for _ in range(12):
    gfu.vec.data = invms * (M.mat * gfu.vec - dt * (C.mat * gfu.vec))
    gfu.AddMultiDimComponent(gfu.vec)
print(f"stir stayed bounded:  T in [{min(gfu.vec):.0f}, {max(gfu.vec):.0f}] °C")
Draw(gfu, mesh, interpolate_multidim=True, animate=True, settings=clip)

That little stir is a first taste of **transport**. In notebook 12 we treat it
properly with a discontinuous-Galerkin scheme — and meet crowds and traffic.

In [ ]:
# Navigation between units — shown only in a live notebook (Colab / JupyterLite /
# local Jupyter), never in the rendered website (which has its own prev/next nav).
import os, sys
if not os.environ.get("WEBGUI_SCENE_DIR"):          # not the static site build
    _prev = ("15-outlook-unfitted", "15 · Outlook — unfitted FEM with ngsxfem 🫧")
    _next = ("17-elasticity", "17 · Making a chocolate bar bend 🍫")
    def _u(_nb):
        if "google.colab" in sys.modules:
            return "https://colab.research.google.com/github/schruste/ngsum2026-colab/blob/colab/" + _nb + ".ipynb"
        return _nb + ".ipynb"                       # JupyterLite & local: relative .ipynb link
    _parts  = ["⬅️ **Previous:** [%s](%s)" % (_prev[1], _u(_prev[0]))] if _prev else []
    _parts += ["➡️ **Next:** [%s](%s)" % (_next[1], _u(_next[0]))] if _next else []
    from IPython.display import display, Markdown
    display(Markdown(" · ".join(_parts)))